# 01 · Companies House exploration

Research log: [Step 1](../docs/research_log.md#step-1--setup-și-prima-privire-la-companies-house), [Step 3](../docs/research_log.md#step-3--primul-caz-real-și-adresele-comune) (in Romanian). Data and checks as of October 2026.

**Question.** What is in the Companies House bulk file, and can it be split into groups that separate "no VAT" from "VAT not found"?

**Method.** DuckDB SQL directly on the 2026-09-01 snapshot (`ignore_errors`, `store_rejects`); counts by accounts category and first SIC code; first manual website check (cell 8, abandoned); shared registered-office addresses.

**Result (from the research log).**
- 5,689,366 rows read in 5.9 s; 1 row rejected (SENTINALL LIMITED, 5 SIC codes vs 4 header columns). There is no VAT column.
- "Group 4" (FULL, MEDIUM, GROUP, AUDIT EXEMPTION SUBSIDIARY) = 155,632 companies.
- Registered-office addresses are heavily shared (61.6% of companies at an address with ≥ 2 companies, 22.9% at ≥ 100), so a postcode match is only a weak signal.

**Notes for re-running.**
- Every cell reads the raw CH file `../data/BasicCompanyDataAsOneFile-2026-09-01.csv` (not in git; see `data/README.md`).
- Cell 8 fetches one live Companies House page.

*Cells are kept as they were run, outputs included; only file paths were updated to the new `data/` layout and session timestamps in outputs were cut to the month.*

In [2]:
import duckdb
duckdb.sql("SELECT * FROM '../data/BasicCompanyDataAsOneFile-2026-09-01.csv' LIMIT 5").df()

,CompanyName,CompanyNumber,RegAddress.CareOf,RegAddress.POBox,RegAddress.AddressLine1,RegAddress.AddressLine2,RegAddress.PostTown,RegAddress.County,RegAddress.Country,RegAddress.PostCode,...,PreviousName_7.CONDATE,PreviousName_7.CompanyName,PreviousName_8.CONDATE,PreviousName_8.CompanyName,PreviousName_9.CONDATE,PreviousName_9.CompanyName,PreviousName_10.CONDATE,PreviousName_10.CompanyName,ConfStmtNextDueDate,ConfStmtLastMadeUpDate
0,! LTD,08209948,None,None,9 PRINCES SQUARE,NaN,HARROGATE,NaN,ENGLAND,HG1 1ND,...,NaT,None,NaT,None,NaT,None,None,None,2026-09-25,2025-09-11
1,!ABRIDGE TAX LTD,16092999,None,None,82 GREAT NORTH ROAD,GREAT NORTH BUSINESS CENTRE,HATFIELD,NaN,UNITED KINGDOM,AL9 5BL,...,NaT,None,NaT,None,NaT,None,None,None,2026-12-04,2025-11-20
2,!BIG IMPACT GRAPHICS LIMITED,11743365,None,None,124 CITY ROAD,NaN,LONDON,NaN,NaN,EC1V 2NX,...,NaT,None,NaT,None,NaT,None,None,None,2026-12-29,2025-12-15
3,!FE NETWORK LIMITED,16873705,None,None,C/O AACSL ACCOUNTANT LIMITED,1ST FLOOR NORTH WESTGATE HOUSE,"HARLOW, ESSEX",NaN,UNITED KINGDOM,CM20 1YS,...,NaT,None,NaT,None,NaT,None,None,None,2026-12-08,NaT
4,!NFLECTION ADVISORY LIMITED,15073164,None,None,74 SANTERS LANE,NaN,POTTERS BAR,HERTFORDSHIRE,ENGLAND,EN6 2DA,...,NaT,None,NaT,None,NaT,None,None,None,2026-08-28,2025-08-14


In [5]:
duckdb.sql("SELECT CompanyStatus FROM '../data/BasicCompanyDataAsOneFile-2026-09-01.csv' LIMIT 5").df()

,CompanyStatus
0,Active
1,Active
2,Active
3,Active
4,Active


In [8]:
duckdb.sql("""
    SELECT companyname, companynumber, regaddressposttown,
           companystatus, accountsaccountcategory
    FROM read_csv(
        '../data/BasicCompanyDataAsOneFile-2026-09-01.csv',
        header = true,
        normalize_names = true,
        all_varchar = true
    )
    USING SAMPLE reservoir(20 ROWS) REPEATABLE (42)
""").df()

InvalidInputException: Invalid Input Error: CSV Error on Line: 4502220
Original Line: "SENTINALL LIMITED","08652773","","","3 THAMES HAVEN","PORTSMOUTH ROAD","SURBITON","SURREY","ENGLAND","KT6 4JA","Private Limited Company","Active","United Kingdom","","15/08/2013","31","8","31/05/2027","31/08/2025","TOTAL EXEMPTION FULL","12/09/2016","15/08/2015","0","0","0","0","56290 - Other food services","62012 - Business and domestic software development","62020 - Information technology consultancy activities","62090 - Other information technology service activities","63110 - Data processing, hosting and related activities","0","0","http://business.data.gov.uk/id/company/08652773","","","","","","","","","","","","","","","","","","","","","29/08/2027","15/08/2026"
Expected Number of Columns: 55 Found: 56
Possible fixes:
* Disable the parser's strict mode (strict_mode=false) to allow reading rows that do not comply with the CSV standard.
* Enable null padding (null_padding=true) to replace missin

In [10]:
duckdb.sql("""
    SELECT COUNT(*) AS rinduri_citite
    FROM read_csv(
        '../data/BasicCompanyDataAsOneFile-2026-09-01.csv',
        header = true,
        normalize_names = true,
        all_varchar = true,
        ignore_errors = true,
        store_rejects = true
    )
""").df()

,rinduri_citite
0,5689366


In [ ]:
duckdb.sql("""
    SELECT  error_type, COUNT(*) AS n
    FROM reject_errors
    GROUP BY error_type
""").df()

,error_type,n
0,TOO MANY COLUMNS,2


In [17]:
duckdb.sql("SELECT COUNT(DISTINCT line) AS rinduri_distincte FROM reject_errors").df()

,rinduri_distincte
0,1


In [18]:
duckdb.sql("SELECT DISTINCT  line, error_type, csv_line FROM reject_errors").df()

,line,error_type,csv_line
0,4502220,TOO MANY COLUMNS,"""SENTINALL LIMITED"",""08652773"","""","""",""3 THAMES..."


In [19]:
duckdb.sql("""
    SELECT companyname, companynumber, regaddressposttown,
           companystatus, accountsaccountcategory
    FROM read_csv(
        '../data/BasicCompanyDataAsOneFile-2026-09-01.csv',
        header = true,
        normalize_names = true,
        all_varchar = true,
        ignore_errors = true
    )
    USING SAMPLE reservoir(20 ROWS) REPEATABLE (42)
""").df()

,companyname,companynumber,regaddressposttown,companystatus,accountsaccountcategory
0,51 UPPERTON GARDENS (EASTBOURNE) LIMITED,09115556,EASTBOURNE,Active,DORMANT
1,151 AND 153 MARLBOROUGH ROAD LIMITED,07893004,LONDON,Active,MICRO ENTITY
2,70 WEBBS ROAD FREEHOLD LIMITED,13436038,LONDON,Active,DORMANT
3,127STAMFORD LIMITED,14702421,MANCHESTER,Active,TOTAL EXEMPTION FULL
4,19 LANSDOWNE STREET (HOVE) LIMITED,05658192,HOVE,Active,DORMANT
5,49 SIMPSON STREET FREEHOLD LIMITED,06195867,LONDON,Active,MICRO ENTITY
6,3SPR MANAGEMENT SERVICES LTD,09537288,BIRMINGHAM,Active,MICRO ENTITY
7,17 LIONGATE GARDENS LIMITED,02328272,SURREY,Active,MICRO ENTITY
8,44 GRAFTON ROAD LTD,16027082,WORTHING,Active,DORMANT
9,20 CABOT SQUARE II UNIT TRUST SLP,SL005921,20 CASTLE TERRACE,Active,NO ACCOUNTS FILED


In [20]:
import requests

url = "https://find-and-update.company-information.service.gov.uk/company/09115556"
raspuns = requests.get(url, timeout=10)

print(raspuns.status_code)
print(len(raspuns.text))

200
35747


In [3]:
import duckdb

duckdb.sql("""
    SELECT accountsaccountcategory, COUNT(*) AS nr_firme
    FROM read_csv('../data/BasicCompanyDataAsOneFile-2026-09-01.csv',
                  normalize_names = true, ignore_errors = true)
    GROUP BY accountsaccountcategory
    ORDER BY nr_firme DESC
""").df()

,accountsaccountcategory,nr_firme
0,MICRO ENTITY,1858859
1,NO ACCOUNTS FILED,1457059
2,TOTAL EXEMPTION FULL,1337628
3,DORMANT,632362
4,UNAUDITED ABRIDGED,165274
5,FULL,86837
6,SMALL,68164
7,AUDIT EXEMPTION SUBSIDIARY,34159
8,GROUP,28258
9,TOTAL EXEMPTION SMALL,8904


In [5]:
duckdb.sql("""
    SELECT siccodesictext_1, COUNT(*) AS nr_firme
    FROM read_csv('../data/BasicCompanyDataAsOneFile-2026-09-01.csv',
                  normalize_names = true, ignore_errors = true)
    GROUP BY siccodesictext_1
    ORDER BY nr_firme DESC
    LIMIT 25
""").df()

,siccodesictext_1,nr_firme
0,68209 - Other letting and operating of own or ...,283055
1,68100 - Buying and selling of own real estate,258268
2,None Supplied,231523
3,70229 - Management consultancy activities othe...,222394
4,82990 - Other business support service activit...,181299
5,47910 - Retail sale via mail order houses or v...,162689
6,62020 - Information technology consultancy act...,129577
7,98000 - Residents property management,126042
8,41100 - Development of building projects,119396
9,96090 - Other service activities n.e.c.,119060


In [7]:
duckdb.sql("""
    SELECT COUNT(DISTINCT siccodesictext_1) AS nr_coduri
    FROM read_csv('../data/BasicCompanyDataAsOneFile-2026-09-01.csv',
                  normalize_names = true, ignore_errors = true)
""").df()

,nr_coduri
0,1164


In [9]:
duckdb.sql("""
    WITH grup4_comercial AS (
        SELECT companyname, companynumber, accountsaccountcategory,
               siccodesictext_1, regaddresspostcode
        FROM read_csv('../data/BasicCompanyDataAsOneFile-2026-09-01.csv',
                      normalize_names = true, ignore_errors = true)
        WHERE accountsaccountcategory IN ('FULL', 'MEDIUM', 'GROUP', 'AUDIT EXEMPTION SUBSIDIARY')
          AND LEFT(siccodesictext_1, 5) NOT IN ('99999', '98000', '68209', '68100', '64209')
          AND siccodesictext_1 <> 'None Supplied'
    )
    SELECT *
    FROM grup4_comercial
    USING SAMPLE reservoir(12 ROWS) REPEATABLE (42)
""").df()

,companyname,companynumber,accountsaccountcategory,siccodesictext_1,regaddresspostcode
0,CAYMANBRACK CAPITAL GROUP PLC,08945805,FULL,82990 - Other business support service activit...,SK4 2LP
1,CREATIVE CARE (EAST MIDLANDS) LIMITED,05522412,FULL,87200 - Residential care activities for learni...,CO4 9PE
2,GREENTECH DISTRIBUTION PLC,07168990,FULL,46520 - Wholesale of electronic and telecommun...,HP10 9RS
3,ARTESIAN SOLUTIONS LIMITED,05667880,GROUP,58290 - Other software publishing,EC4A 3HQ
4,DSV LEAD LOGISTICS LIMITED,03421165,FULL,49390 - Other passenger land transport,CO12 4QG
5,FINANCIAL SERVICES GROUP LIMITED,07031955,GROUP,64999 - Financial intermediation not elsewhere...,TN15 8AD
6,ELLIOTT WESTLAND INSURANCE BROKERS LIMITED,04115071,AUDIT EXEMPTION SUBSIDIARY,65120 - Non-life insurance,HP13 6NU
7,FLOW HOSPITALITY TRAINING LIMITED,SC354749,FULL,"74909 - Other professional, scientific and tec...",EH12 5HD
8,ASG PRINT HOLDINGS LIMITED,09242996,AUDIT EXEMPTION SUBSIDIARY,64202 - Activities of production holding compa...,W14 0HN
9,GD FIRST (UK) LIMITED,04955958,FULL,70100 - Activities of head offices,BD4 0SG


In [10]:
duckdb.sql("""
    WITH grup4_comercial AS (
        SELECT companyname, companynumber, accountsaccountcategory,
               siccodesictext_1, regaddresspostcode
        FROM read_csv('../data/BasicCompanyDataAsOneFile-2026-09-01.csv',
                      normalize_names = true, ignore_errors = true)
        WHERE accountsaccountcategory IN ('FULL', 'MEDIUM', 'GROUP', 'AUDIT EXEMPTION SUBSIDIARY')
          AND LEFT(siccodesictext_1, 5) NOT IN ('99999', '98000', '68209', '68100', '64209')
          AND siccodesictext_1 <> 'None Supplied'
    )
    SELECT *
    FROM grup4_comercial
    USING SAMPLE reservoir(12 ROWS) REPEATABLE (42)
""").df().to_csv('../data/samples/set_explorare_grup4.csv', index=False)

In [2]:
import duckdb

duckdb.sql("""
    SELECT companyname, companynumber, regaddresspostcode,
           accountsaccountcategory, siccodesictext_1
    FROM read_csv('../data/BasicCompanyDataAsOneFile-2026-09-01.csv',
                  normalize_names = true, ignore_errors = true)
    WHERE companynumber = '09325294'
""").df()

,companyname,companynumber,regaddresspostcode,accountsaccountcategory,siccodesictext_1
0,BLOCKCHAIN (GB) LIMITED,09325294,SK4 2LP,GROUP,62012 - Business and domestic software develop...


In [3]:
duckdb.sql("""
    SELECT companyname, companynumber, accountsaccountcategory, siccodesictext_1
    FROM read_csv('../data/BasicCompanyDataAsOneFile-2026-09-01.csv',
                  normalize_names = true, ignore_errors = true)
    WHERE regaddresspostcode = 'SK4 2LP'
""").df()

,companyname,companynumber,accountsaccountcategory,siccodesictext_1
0,13 HFR LIMITED,14985239,MICRO ENTITY,98000 - Residents property management
1,24-7 GAS SERVICES LIMITED,16583688,NO ACCOUNTS FILED,"43220 - Plumbing, heat and air-conditioning in..."
2,A I SECURITY LIMITED,13079135,MICRO ENTITY,80100 - Private security activities
3,A&S ENVIRONMENTAL LIMITED,16830368,NO ACCOUNTS FILED,38110 - Collection of non-hazardous waste
4,AAZAMA LTD,17244892,NO ACCOUNTS FILED,68320 - Management of real estate on a fee or ...
...,...,...,...,...
214,WILLOW PARK (FALLOWFIELD) MANAGEMENT CO. LIMITED,01299593,MICRO ENTITY,81100 - Combined facilities support activities
215,WYCH PROPERTIES LTD,17167639,NO ACCOUNTS FILED,68209 - Other letting and operating of own or ...
216,XEN-GLOW LTD,15587478,TOTAL EXEMPTION FULL,32990 - Other manufacturing n.e.c.
217,XIAN NETWORKS LIMITED,05573162,MICRO ENTITY,82990 - Other business support service activit...


In [4]:
prag = 10

duckdb.sql(f"""
    WITH adrese AS (
        SELECT regaddresspostcode, regaddressaddressline1,
               COUNT(*) AS nr_firme
        FROM read_csv('../data/BasicCompanyDataAsOneFile-2026-09-01.csv',
                      normalize_names = true, ignore_errors = true)
        WHERE regaddresspostcode IS NOT NULL
        GROUP BY regaddresspostcode, regaddressaddressline1
    )
    SELECT
        COUNT(*) FILTER (WHERE nr_firme >= {prag})            AS adrese_comune,
        SUM(nr_firme) FILTER (WHERE nr_firme >= {prag})       AS firme_la_adrese_comune,
        SUM(nr_firme)                                          AS firme_total,
        ROUND(100.0 * SUM(nr_firme) FILTER (WHERE nr_firme >= {prag})
              / SUM(nr_firme), 1)                              AS procent
    FROM adrese
""").df()

,adrese_comune,firme_la_adrese_comune,firme_total,procent
0,34860,2115327.0,5598126.0,37.8


In [5]:
prag = 2

duckdb.sql(f"""
    WITH adrese AS (
        SELECT regaddresspostcode, regaddressaddressline1,
               COUNT(*) AS nr_firme
        FROM read_csv('../data/BasicCompanyDataAsOneFile-2026-09-01.csv',
                      normalize_names = true, ignore_errors = true)
        WHERE regaddresspostcode IS NOT NULL
        GROUP BY regaddresspostcode, regaddressaddressline1
    )
    SELECT
        COUNT(*) FILTER (WHERE nr_firme >= {prag})            AS adrese_comune,
        SUM(nr_firme) FILTER (WHERE nr_firme >= {prag})       AS firme_la_adrese_comune,
        SUM(nr_firme)                                          AS firme_total,
        ROUND(100.0 * SUM(nr_firme) FILTER (WHERE nr_firme >= {prag})
              / SUM(nr_firme), 1)                              AS procent
    FROM adrese
""").df()

,adrese_comune,firme_la_adrese_comune,firme_total,procent
0,525503,3449935.0,5598126.0,61.6


In [6]:
prag = 100

duckdb.sql(f"""
    WITH adrese AS (
        SELECT regaddresspostcode, regaddressaddressline1,
               COUNT(*) AS nr_firme
        FROM read_csv('../data/BasicCompanyDataAsOneFile-2026-09-01.csv',
                      normalize_names = true, ignore_errors = true)
        WHERE regaddresspostcode IS NOT NULL
        GROUP BY regaddresspostcode, regaddressaddressline1
    )
    SELECT
        COUNT(*) FILTER (WHERE nr_firme >= {prag})            AS adrese_comune,
        SUM(nr_firme) FILTER (WHERE nr_firme >= {prag})       AS firme_la_adrese_comune,
        SUM(nr_firme)                                          AS firme_total,
        ROUND(100.0 * SUM(nr_firme) FILTER (WHERE nr_firme >= {prag})
              / SUM(nr_firme), 1)                              AS procent
    FROM adrese
""").df()

,adrese_comune,firme_la_adrese_comune,firme_total,procent
0,4033,1283031.0,5598126.0,22.9


In [7]:
duckdb.sql("""
    SELECT regaddresspostcode, regaddressaddressline1,
           COUNT(*) AS nr_firme
    FROM read_csv('../data/BasicCompanyDataAsOneFile-2026-09-01.csv',
                  normalize_names = true, ignore_errors = true)
    WHERE regaddresspostcode IS NOT NULL
    GROUP BY regaddresspostcode, regaddressaddressline1
    ORDER BY nr_firme DESC
    LIMIT 20
""").df()

,regaddresspostcode,regaddressaddressline1,nr_firme
0,WC2H 9JQ,71-75 SHELTON STREET,86448
1,EC1V 2NX,128 CITY ROAD,42746
2,W1W 5PF,167-169 GREAT PORTLAND STREET,23525
3,N1 7GU,20 WENLOCK ROAD,20477
4,EC1V 2NX,124 CITY ROAD,17023
5,N1 7GU,20-22 WENLOCK ROAD,15383
6,EC2A 4NE,"3RD FLOOR, 86-90",13559
7,WC1N 3AX,27 OLD GLOUCESTER STREET,13388
8,EC2A 4NA,66 PAUL STREET,10191
9,EC2A 4NE,86-90 PAUL STREET,9091


In [10]:
duckdb.sql("""
    DESCRIBE SELECT *
    FROM read_csv('../data/BasicCompanyDataAsOneFile-2026-09-01.csv',
                  normalize_names = true, ignore_errors = true)
""").df()

,column_name,column_type,null,key,default,extra
0,companyname,VARCHAR,YES,None,None,None
1,companynumber,VARCHAR,YES,None,None,None
2,regaddresscareof,VARCHAR,YES,None,None,None
3,regaddresspobox,VARCHAR,YES,None,None,None
4,regaddressaddressline1,VARCHAR,YES,None,None,None
5,regaddressaddressline2,VARCHAR,YES,None,None,None
6,regaddressposttown,VARCHAR,YES,None,None,None
7,regaddresscounty,VARCHAR,YES,None,None,None
8,regaddresscountry,VARCHAR,YES,None,None,None
9,regaddresspostcode,VARCHAR,YES,None,None,None


In [11]:
duckdb.sql("""
    SELECT companycategory, countryoforigin, regaddresscountry,
           COUNT(*) AS nr_firme
    FROM read_csv('../data/BasicCompanyDataAsOneFile-2026-09-01.csv',
                  normalize_names = true, ignore_errors = true)
    WHERE regaddresspostcode IS NULL
    GROUP BY ALL
    ORDER BY nr_firme DESC
    LIMIT 15
""").df()

,companycategory,countryoforigin,regaddresscountry,nr_firme
0,Charitable Incorporated Organisation,United Kingdom,NaN,40416
1,Registered Society,United Kingdom,NaN,10811
2,Scottish Charitable Incorporated Organisation,United Kingdom,NaN,7923
3,Limited Partnership,United Kingdom,NaN,6037
4,Overseas Entity,"VIRGIN ISLANDS, BRITISH","VIRGIN ISLANDS, BRITISH",4035
5,Private Limited Company,United Kingdom,NaN,2534
6,Other company type,UNITED STATES,UNITED STATES,2083
7,Other company type,United Kingdom,NaN,930
8,Royal Charter Company,United Kingdom,NaN,909
9,Other company type,IRELAND,IRELAND,773


In [12]:
duckdb.sql("""
    SELECT accountsaccountcategory,
           COUNT(*) AS total,
           COUNT(*) FILTER (WHERE companycategory = 'Public Limited Company') AS plc_dupa_categorie,
           COUNT(*) FILTER (WHERE UPPER(companyname) LIKE '%PLC%')         AS plc_dupa_nume
    FROM read_csv('../data/BasicCompanyDataAsOneFile-2026-09-01.csv',
                  normalize_names = true, ignore_errors = true)
    WHERE accountsaccountcategory IN ('FULL','MEDIUM','GROUP','AUDIT EXEMPTION SUBSIDIARY')
    GROUP BY accountsaccountcategory
    ORDER BY total DESC
""").df()

,accountsaccountcategory,total,plc_dupa_categorie,plc_dupa_nume
0,FULL,86837,2101,1983
1,AUDIT EXEMPTION SUBSIDIARY,34159,38,36
2,GROUP,28258,1505,1441
3,MEDIUM,6378,6,6
